In [0]:
from pyspark.sql.functions import col, lit, udf, to_timestamp, lpad, concat_ws, when, date_add, count, isnull, expr

In [0]:
base_flights_path = "workspace.flights.t_ontime_reporting_{}"
month_list = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
for month in month_list:
    month_path = base_flights_path.format(month)
    df = spark.table(month_path)
    if month == "jan":
        df_FlightsData = df
    else:
        df_FlightsData = df_FlightsData.unionByName(df)
df_FlightsData.count()

In [0]:
df_FlightsData.head()

In [0]:
df_FilteredFlights = df_FlightsData.filter((col("CANCELLED")==0) & (col("DIVERTED")==0))
df_FilteredFlights.count()

In [0]:
df_NullFilteredFlights = df_FilteredFlights.dropna(how="any", subset=["DEP_DELAY", "ARR_DELAY", "DEP_TIME", "ARR_TIME", "CRS_DEP_TIME", "CRS_ARR_TIME", "CRS_ELAPSED_TIME"])
df_NullFilteredFlights.count()

In [0]:
def create_timestamp(date_col, time_col):
    date_fmt = "M/d/yyyy h:mm:ss a"
    date_only = to_timestamp(col(date_col), date_fmt)
    
    adjusted_time = when(col(time_col) == 2400, 0).otherwise(col(time_col))
    add_day = when(col(time_col) == 2400, 1).otherwise(0)
    
    padded_time = lpad(adjusted_time.cast("int").cast("string"), 4, '0')
    hour = padded_time.substr(1, 2)
    minute = padded_time.substr(3, 2)
    datetime_str = concat_ws(' ', date_add(date_only.cast("date"), add_day), concat_ws(':', hour, minute))
    return datetime_str.cast("timestamp")

In [0]:
df_NullFilteredFlights.display()

In [0]:
# 1. Parse the flight date
df = df_NullFilteredFlights.withColumn("flight_date", to_timestamp(col("FL_DATE"), "M/d/yyyy h:mm:ss a"))

# 2. Create the Scheduled Timestamps using your custom function
df = df.withColumn("schedule_dep", create_timestamp("FL_DATE", "CRS_DEP_TIME")) \
       .withColumn("schedule_arr_temp", create_timestamp("FL_DATE", "CRS_ARR_TIME"))

# 3. Handle Red-Eye Flights 
# (If Scheduled Arrival Time is smaller than Scheduled Departure Time, the flight lands the next day)
df = df.withColumn(
    "schedule_arr",
    when(col("CRS_ARR_TIME") < col("CRS_DEP_TIME"), 
         expr("from_unixtime(unix_timestamp(schedule_arr_temp) + 86400)").cast("timestamp") # 86400 seconds = 1 day
    ).otherwise(col("schedule_arr_temp"))
).drop("schedule_arr_temp")

# 4. Calculate Actual Times Mathematically Using Delays (Protects against midnight rollovers!)
# Delay is in minutes, so multiply by 60 for seconds.
df_FinalFlights = df.withColumn(
    "actual_dept", 
    expr("from_unixtime(unix_timestamp(schedule_dep) + (DEP_DELAY * 60))").cast("timestamp")
).withColumn(
    "actual_arr", 
    expr("from_unixtime(unix_timestamp(schedule_arr) + (ARR_DELAY * 60))").cast("timestamp")
)

# 5. Safely display the result
display(df_FinalFlights)

In [0]:
# 1. Alias the tables so column names don't collide (e.g., ORIGIN_A vs ORIGIN_B)
df_A = df_FinalFlights.select([col(c).alias(c + "_A") for c in df_FinalFlights.columns])
df_B = df_FinalFlights.select([col(c).alias(c + "_B") for c in df_FinalFlights.columns])

# 2. Execute the Optimized Self-Join
connections = df_A.join(
    df_B,
    (col("DEST_A") == col("ORIGIN_B")) &  # Same Hub
    (col("flight_date_A") == col("flight_date_B")) &  # SAME DAY OPTIMIZATION
    # Scheduled layover must be between 30 mins (1800 sec) and 6 hours (21600 sec)
    (expr("unix_timestamp(schedule_dep_B) - unix_timestamp(schedule_arr_A) >= 1800")) & 
    (expr("unix_timestamp(schedule_dep_B) - unix_timestamp(schedule_arr_A) <= 21600"))
)

# 3. Create the Target Variable ('made_connection') & Core Feature ('scheduled_layover_mins')
# We assume the passenger needs at least 30 minutes (1800 seconds) physically in the airport to walk to the next gate
connections = connections.withColumn(
    "scheduled_layover_mins",
    expr("(unix_timestamp(schedule_dep_B) - unix_timestamp(schedule_arr_A)) / 60")
).withColumn(
    "made_connection",
    # Target Variable: 1 if (Actual Arrival A + 30 mins) <= Actual Departure B. Else 0.
    when(expr("unix_timestamp(actual_arr_A) + 1800 <= unix_timestamp(actual_dept_B)"), 1).otherwise(0)
)

# 4. Check the results!
print(f"Total historical connections generated: {connections.count()}")

display(connections.select(
    "OP_UNIQUE_CARRIER_A", "ORIGIN_A", "DEST_A", "DEST_B", 
    "schedule_dep_A", "schedule_dep_B", 
    "scheduled_layover_mins", "made_connection"
).limit(10))

In [0]:
from pyspark.sql.functions import col, when, hour

# 1. Feature 1: Is it the same airline? (1 for Yes, 0 for No)
connections = connections.withColumn(
    "is_same_airline",
    when(col("OP_UNIQUE_CARRIER_A") == col("OP_UNIQUE_CARRIER_B"), 1).otherwise(0)
)

# 2. Feature 2: Hour of the Day at the Hub (captures congestion)
connections = connections.withColumn(
    "hub_congestion_hour",
    hour(col("schedule_dep_B"))
)

# 3. Feature 3: Month of the year (captures seasonality directly)
connections = connections.withColumn(
    "travel_month",
    col("flight_date_A").cast("string").substr(6, 2).cast("int") # extracts month from YYYY-MM-DD
)

# --- THE DOWNSAMPLING STRATEGY ---
# Let's check the imbalance first (this will take a minute to run on 599M rows)
display(connections.groupBy("made_connection").count())

In [0]:
# To prevent MLlib from crashing, we sample the data. 
# We take 1% of the majority class (Success) and a higher percentage of the minority class (Missed) 
# to balance the dataset. Let's do a simple 1% random sample to get ~5 million rows for fast training.

ml_ready_df = connections.sample(withReplacement=False, fraction=0.01, seed=42)

# Keep only the columns the Machine Learning model needs
ml_ready_df = ml_ready_df.select(
    "is_same_airline",
    "hub_congestion_hour",
    "travel_month",
    "scheduled_layover_mins",
    "made_connection" # The Target
)

print(f"Final ML Training Dataset Size: {ml_ready_df.count()}")
display(ml_ready_df.limit(5))